# Plan 04: the injection probe (paper Figure 1)

Base against Base + **one** injection template at a time, on rel-f1's three tasks. This repeats the paper's probe experiment (Section 3.3, Figure 1):

| Setting | What it adds |
|---|---|
| `base` | nothing (RelGraphSAGE) |
| `inject-knn`, n = 5 / 10 | each driver receives from its n most similar drivers (earlier seeds only) |
| `inject-temporal`, m = 1 / 5 | each record linked to its previous m records of the same driver, constructor or circuit |
| `inject-twohop` | a shortcut for every path A → B → C with A ≠ C (66 relations) |
| `inject-behavioral` | drivers with similar results / standings / qualifying histories (similarity > 0.9) |

**Settings:**
- RelBench's settings: 10 epochs, lr 0.005, hidden 128;
- `--dedup-encoder` and `--nan-safe-numerical`, as in Plan 01;
- 5 seeds per setting, 35 runs per task;
- one more config times KNN on every table against drivers only (seconds per epoch).

The paper's pattern, which is what we look for: KNN helps classification (`driver-top3`, `driver-dnf`); temporal continuity helps regression (`driver-position`). Settings are compared on **validation**; test is shown for reference, next to the paper's.

**On Colab:**
1. *Runtime → Change runtime type → T4 GPU*. The first cell says which device it found.
2. *Runtime → Run all*. About half an hour on a T4 (an estimate from Plan 01's timings).
3. If Colab disconnects, run all again: with `USE_DRIVE = True`, finished runs are kept in Drive and skipped.
4. The last cell downloads a zip of all four results folders. Upload it, and this notebook saved with outputs (*File → Download → .ipynb*).


In [ ]:
# ---- Settings ----
PROJECT_REPO = 'https://github.com/HARSHALK2598/cs224w-structural-optimizer'
PROJECT_BRANCH = 'step/04-2-train-injection'     # train.py --inject, the probe configs
PYG_REPO = 'https://github.com/HARSHALK2598/pytorch_geometric'
PYG_BRANCH = 'step/04-1-injection-templates'     # KNNEdges, TwoHopEdges, ...
RELBENCH_COMMIT = '91c1f6420a03feb61c279aaeb194d7268435f278'  # as in scripts/setup_env.sh

SWEEP_CONFIGS = [
    'experiments/sweeps/probe_injection_driver-top3.yaml',
    'experiments/sweeps/probe_injection_driver-dnf.yaml',
    'experiments/sweeps/probe_injection_driver-position.yaml',
    'experiments/sweeps/probe_knn_cost_driver-top3.yaml',
]
RUN_QUICK_CHECK = True   # 2 runs x 1 epoch of the first config, before the sweeps
REQUIRE_GPU = False      # True: stop if there is no GPU
USE_DRIVE = True         # Colab only: keep results in Google Drive
DRIVE_DIR = '/content/drive/MyDrive/cs224w-structural-optimizer'


## 1. Environment

The same setup as `colab_sweep.ipynb`: on Colab, clones both repositories and installs pinned versions; elsewhere, uses the current environment.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules


def sh(cmd, cwd=None):
    """Runs a shell command, streaming its output; stops on failure."""
    print('$', cmd, flush=True)
    # PYTHONUNBUFFERED: show the sweep's lines as they happen, not at the end.
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed ({proc.returncode}): {cmd}')


HAS_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(
    ['nvidia-smi'], capture_output=True).returncode == 0
if HAS_GPU:
    print('=' * 70 + '\nRUNNING ON GPU\n' + '=' * 70, flush=True)
else:
    print('=' * 70 + '\nRUNNING ON CPU: no GPU found. Training will be much slower.\n'
          'For a GPU: Runtime -> Change runtime type -> T4 GPU, then run all again.\n'
          + '=' * 70, flush=True)
    if REQUIRE_GPU:
        raise RuntimeError('REQUIRE_GPU is set and there is no GPU')

if IN_COLAB:
    ROOT = Path('/content/cs224w-structural-optimizer')
    if not ROOT.exists():
        sh(f'git clone -q --branch {PROJECT_BRANCH} {PROJECT_REPO} {ROOT}')
    else:
        sh(f'git -C {ROOT} pull -q origin {PROJECT_BRANCH}')
    pyg = ROOT / 'pytorch_geometric'
    if not pyg.exists():
        sh(f'git clone -q --branch {PYG_BRANCH} {PYG_REPO} {pyg}')
    else:
        sh(f'git -C {pyg} pull -q origin {PYG_BRANCH}')

    build = 'cu126' if HAS_GPU else 'cpu'
    pip = f'{sys.executable} -m pip install -q'
    sh(f'{sys.executable} -m pip uninstall -q -y torchvision torchaudio')
    sh(f'{pip} torch==2.12.0 --index-url https://download.pytorch.org/whl/{build}')
    sh(f'{pip} pyg_lib==0.9.0 -f https://data.pyg.org/whl/torch-2.12.0+{build}.html')
    sh(f'{pip} -e {pyg}')
    sh(f'{pip} "relbench @ git+https://github.com/snap-stanford/relbench@{RELBENCH_COMMIT}" '
       f'pytorch-frame==0.3.0 sentence-transformers==6.2.0 pyyaml pandas')
else:
    # Run from anywhere inside the project repository (or set PROJECT_ROOT).
    ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd())).resolve()
    while not (ROOT / 'experiments' / 'sweep.py').exists():
        if ROOT.parent == ROOT:
            raise RuntimeError('Start the notebook inside the project repository.')
        ROOT = ROOT.parent

os.chdir(ROOT)
print('project root:', ROOT, '| Colab:', IN_COLAB, '| GPU:', HAS_GPU)
sh(f'{sys.executable} -c "import torch, torch_geometric, torch_frame, relbench, pyg_lib; '
   f'print(\'torch\', torch.__version__, \'| cuda\', torch.cuda.is_available(), '
   f'torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'\', '
   f'\'| pyg\', torch_geometric.__version__, \'| frame\', torch_frame.__version__, '
   f'\'| relbench\', relbench.__version__, \'| pyg_lib\', pyg_lib.__version__)"')
sh('git log --oneline -1')
sh('git -C pytorch_geometric log --oneline -1' if (ROOT / 'pytorch_geometric').exists() else 'true')

## 2. Where results go

One folder per config, `sweeps/<name>/`: in Google Drive on Colab with `USE_DRIVE = True`, in the repository otherwise.

In [ ]:
import yaml

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path(DRIVE_DIR) / 'sweeps'
else:
    BASE = ROOT / 'experiments' / 'results' / 'sweeps'
CONFIGS = {}
for path in SWEEP_CONFIGS:
    with open(ROOT / path) as f:
        cfg = yaml.safe_load(f)
    CONFIGS[path] = (cfg, BASE / cfg['name'])
    (BASE / cfg['name']).mkdir(parents=True, exist_ok=True)
    print(cfg['name'], '->', BASE / cfg['name'])


## 3. Quick check: 2 runs × 1 epoch

Into a separate `check/` folder, so it never mixes with the results.

In [ ]:
if RUN_QUICK_CHECK:
    path, (cfg, out) = next(iter(CONFIGS.items()))
    check = out / 'check'
    shutil.rmtree(check, ignore_errors=True)
    sh(f'{sys.executable} experiments/sweep.py {path} --max-runs 2 --epochs 1 --out "{check}"')


## 4. The sweeps

One line per finished run. Rerunning this cell continues where it stopped.

In [ ]:
for path, (cfg, out) in CONFIGS.items():
    print(f'\n======== {cfg["name"]} ========', flush=True)
    sh(f'{sys.executable} experiments/sweep.py {path} --out "{out}"')


## 5. Results

For each task, every setting with mean ± std over seeds: validation and test (ROC-AUC for classification; MAE for `driver-position`, where lower is better), and minutes per run. Then the KNN cost check.

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)

for path, (cfg, out) in CONFIGS.items():
    if not (out / 'results.csv').exists():
        continue
    res = pd.read_csv(out / 'results.csv')
    res['setting'] = res['run_id'].str.replace(r'_seed\d+$', '', regex=True)
    metric = 'roc_auc' if res['val_roc_auc'].notna().any() else 'mae'
    table = res.groupby('setting').agg(
        seeds=('seed', 'count'),
        val_mean=(f'val_{metric}', 'mean'), val_std=(f'val_{metric}', 'std'),
        test_mean=(f'test_{metric}', 'mean'), test_std=(f'test_{metric}', 'std'),
        relations=('injected_relations', 'max'), minutes=('minutes', 'mean'))
    table = table.sort_values('val_mean', ascending=(metric == 'mae'))
    print(f"\n{cfg['name']} ({metric}, {'lower' if metric == 'mae' else 'higher'} is better, "
          f"best validation first; {len(res)} runs on {res['device'].iloc[0]})")
    print(table.round(4).to_string())
    failures = out / 'failures.log'
    if failures.exists():
        print('FAILURES:', failures.read_text()[:2000])


## 6. Download the results

In [ ]:
import tempfile
zip_dir = Path('/content') if IN_COLAB else Path(tempfile.gettempdir())
bundle = zip_dir / 'probe_injection'
shutil.rmtree(bundle, ignore_errors=True)
for cfg, out in CONFIGS.values():
    if out.exists():
        shutil.copytree(out, bundle / cfg['name'])
archive = shutil.make_archive(str(bundle), 'zip', root_dir=bundle)
print('wrote', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
